# Retrieval Candidate-Generation Ablation Study

This notebook evaluates four listwise reranking settings while keeping the reranker, losses, folds, label representations, and optimization settings fixed:

1. Ranked full label set (without retrieval)
2. Ranked with ISO + kNN retrieval
3. Ranked with ISO-only retrieval
4. Ranked with kNN-only retrieval


In [ ]:
# =========================
# 1) Imports
# =========================
import os
import re
import gc
import json
import math
import copy
import random
import shutil
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    classification_report,
)

try:
    from sentence_transformers import SentenceTransformer
except Exception as e:
    SentenceTransformer = None
    print("WARNING: sentence-transformers is not installed. Install it before running this file.")
    print(e)

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
    set_seed,
)

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = lambda x, **kwargs: x

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive was mounted automatically.')
except Exception as e:
    print('Google Drive was not mounted automatically:', e)

In [ ]:
# =========================
# 2) Configuration
# =========================
@dataclass
class CFG:
    # Reproducibility
    SEED: int = 42
    N_FOLDS: int = 5

    # Prepared dataset with the exact zero-based test-fold assignment used by all experiments.
    CV_DATASET_PATH: str = "/data/dataset_with_test_fold.xlsx"

    # Original ISO file is still used to read L1/L2 mapping and fallback definitions.
    ISO_CARD_PATH: str = "/content/drive/MyDrive/Colab Notebooks/PhDThesis-Anno-June/Model Building/Corpus /iso.xlsx"

    # Improved full contrastive card file.
    # Upload this file to /content or change this path to the Drive location.
    USE_EXTERNAL_FULL_CONTRASTIVE_CARDS: bool = True
    FULL_CARD_PATH: str = "/documentation/iso_label_full_cards.csv"
    FULL_CARD_SHEET: str = "Improved_Cards"
    FULL_CARD_COL: str = "full_contrastive_label_card"

    # Fallback filenames searched automatically if FULL_CARD_PATH is wrong.
    FULL_CARD_FALLBACK_FILENAMES: Tuple[str, ...] = (
        "iso_label_cards_full_contrastive.xlsx",
    )

    # Base output folder for all ablations
    BASE_OUT_DIR: str = "/results/05_reiso_nfr_iso_only"
    OUT_DIR: str = BASE_OUT_DIR

    #"full_label","iso_plus_knn", "iso_only","knn_only",
    RUN_ABLATIONS: Tuple[str, ...] = (
        "full_label",
    )
    SKIP_COMPLETED_ABLATIONS: bool = True

    # Column names. Leave as None to auto-detect.
    TEXT_COL: Optional[str] = None
    L1_COL: Optional[str] = None
    L2_COL: Optional[str] = None

    # Fallback card mode if an external full card is missing.
    # Options: signals_only, definition_only, definition_plus_signals, signals_plus_sibling, full
    CARD_TEXT_MODE: str = "definition_plus_signals"

    # Reranker card:
    # The retriever uses FULL_CARD_COL from the improved full contrastive workbook.
    # The reranker uses shorter definition+signals cards. By default these are rebuilt from ISO_CARD_PATH.
    # Optionally, set USE_EXTERNAL_RERANKER_SHORT_CARDS=True to load a saved previous card sheet/file.
    USE_EXTERNAL_RERANKER_SHORT_CARDS: bool = True
    RERANKER_SHORT_CARD_PATH_OR_URL: str = "/documentation/iso_label_short_cards.csv"
    RERANKER_SHORT_CARD_SHEET: Optional[str] = None
    RERANKER_SHORT_CARD_COL_CANDIDATES: Tuple[str, ...] = (
        "label_card",
        "definition_plus_signals",
        "old_definition_plus_signals_card",
        "short_label_card",
        "reranker_card",
    )

    # Retriever: Qwen3
    RETRIEVER_MODEL_NAME: str = "Qwen/Qwen3-Embedding-0.6B"
    RETRIEVAL_BATCH_SIZE: int = 4
    USE_QWEN3_INSTRUCTION: bool = True

    # Query-side instruction used when encoding requirement texts.
    QWEN3_QUERY_INSTRUCTION: str = (
        "Retrieve the ISO/IEC 25010 quality sub-characteristic or similar labeled software "
        "requirements that best match this software requirement."
    )

    # Corpus/document-side instruction. Keep empty by default.
    QWEN3_DOC_INSTRUCTION: str = ""

    KNN_TOP_N: int = 20
    CANDIDATE_K: int = 10
    ISO_WEIGHT: float = 0.50
    KNN_WEIGHT: float = 0.50
    KNN_SCORE_MODE: str = "max"      # max, mean, sum
    RETRIEVAL_RECALL_KS: Tuple[int, ...] = (1, 3, 5, 10)

    # Hierarchy-guided hard negatives for TRAINING ONLY.
    USE_HARD_SIBLING_NEGATIVES: bool = True
    N_HARD_SIBLING_NEGATIVES: int = 3
    HARD_SIBLING_STRATEGY: str = "mixed"  # mixed = keep retrieved siblings first, then sample missing siblings
    HARD_SIBLING_SEED_OFFSET: int = 10000

    # Re-ranker model
    RERANKER_MODEL_NAME: str = "microsoft/deberta-v3-base"

    # Full cards are long. Keep 256 first for fair comparison and memory safety.
    # You may test 384 later if Colab memory allows.
    MAX_LEN: int = 256

    # Dropout regularization
    HIDDEN_DROPOUT: float = 0.10
    ATTENTION_DROPOUT: float = 0.10
    CLASSIFIER_DROPOUT: float = 0.10

    # Training
    EPOCHS: int = 30
    PATIENCE: int = 3
    LR: float = 2e-5
    WEIGHT_DECAY: float = 0.01
    WARMUP_RATIO: float = 0.06

    TRAIN_BATCH_SIZE: int = 2      # 4 requirements x 10 candidates = 40 pair inputs
    EVAL_BATCH_SIZE: int = 4
    MAX_GRAD_NORM: float = 1.0
    FP16: bool = False  # safer for DeBERTa reranker; Qwen3 retriever can still use fp16

    # Optional weighted listwise CE by gold L2 label.
    USE_CLASS_WEIGHTS: bool = True
    CLASS_WEIGHT_CLIP: float = 5.0

    # L1 auxiliary objective for listwise reranking.
    # It aggregates candidate L2 scores into parent/L1 logits and adds:
    # total_loss = L2_listwise_CE + LAMBDA_L1_AUX * L1_parent_CE
    USE_L1_AUX: bool = False
    LAMBDA_L1_AUX: float = 0.00
    L1_AUX_POOL: str = "logmeanexp"  # max, mean, logsumexp, logmeanexp

    # Hierarchy-aware margin ranking loss.
    # This adds a hinge penalty when a negative candidate score is too close to or above the gold L2 score.
    # Sibling negatives under the same L1 parent receive a stricter margin than cross-parent negatives.
    USE_SIBLING_MARGIN_LOSS: bool = True
    LAMBDA_SIBLING_MARGIN: float = 0.05
    MARGIN_SIBLING: float = 0.20
    MARGIN_GLOBAL: float = 0.05
    SIBLING_MARGIN_INCLUDE_GLOBAL_NEGATIVES: bool = True

    # Save models. Set False if Drive storage is limited.
    SAVE_BEST_MODEL: bool = True
    DELETE_FOLD_MODEL_AFTER_EVAL: bool = False


cfg = CFG()
Path(cfg.BASE_OUT_DIR).mkdir(parents=True, exist_ok=True)

print("CV_DATASET_PATH:", cfg.CV_DATASET_PATH)
print("OUT_DIR:", cfg.OUT_DIR)
print("Retriever:", cfg.RETRIEVER_MODEL_NAME)
print("Re-ranker:", cfg.RERANKER_MODEL_NAME)
print("Ablations selected:", cfg.RUN_ABLATIONS)
print("External full cards:", cfg.USE_EXTERNAL_FULL_CONTRASTIVE_CARDS)
print("FULL_CARD_PATH:", cfg.FULL_CARD_PATH)
print("FULL_CARD_COL:", cfg.FULL_CARD_COL)
print("Qwen3 instruction:", cfg.USE_QWEN3_INSTRUCTION)
print("Card fallback mode:", cfg.CARD_TEXT_MODE)
print("Hard sibling negatives:", cfg.USE_HARD_SIBLING_NEGATIVES, "n=", cfg.N_HARD_SIBLING_NEGATIVES)
print("L1 auxiliary loss:", cfg.USE_L1_AUX, "lambda=", cfg.LAMBDA_L1_AUX, "pool=", cfg.L1_AUX_POOL)
print("Sibling margin loss:", cfg.USE_SIBLING_MARGIN_LOSS, "lambda=", cfg.LAMBDA_SIBLING_MARGIN, "m_sib=", cfg.MARGIN_SIBLING, "m_global=", cfg.MARGIN_GLOBAL, "include_global=", cfg.SIBLING_MARGIN_INCLUDE_GLOBAL_NEGATIVES)


### Select the experiment here

Use only `RUN_ABLATIONS` in the configuration cell. There is no separate `MODE` setting.

```python
RUN_ABLATIONS = ("full_label",)       # rerank all labels; retrieval is skipped
RUN_ABLATIONS = ("iso_plus_knn",)     # ISO + kNN retrieval
RUN_ABLATIONS = ("iso_only",)         # ISO-only retrieval
RUN_ABLATIONS = ("knn_only",)         # kNN-only retrieval
```

A one-item tuple must retain the trailing comma.


In [ ]:
# =========================
# 3) General utilities
# =========================
def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    set_seed(seed)


def safe_cell(x: Any) -> str:
    if pd.isna(x):
        return ""
    s = str(x).strip()
    if s.lower() in {"nan", "none", "null"}:
        return ""
    s = s.replace("\n", " ").replace("\t", " ")
    s = re.sub(r"\s+", " ", s)
    return s.strip()


def clean_text(s: Any) -> str:
    return safe_cell(s)


def norm_label(s: Any) -> str:
    """Normalize labels to internal IDs such as functionalsuitability."""
    s = safe_cell(s).lower()
    s = s.replace("&", "and")
    s = re.sub(r"[^a-z0-9]+", "", s)
    return s


def pretty_label_from_id(label_id: str) -> str:
    """Fallback only. Prefer L1_name/L2_name from ISO Excel."""
    replacements = {
        "functionalsuitability": "functional suitability",
        "functionalcompleteness": "functional completeness",
        "functionalcorrectness": "functional correctness",
        "functionalappropriateness": "functional appropriateness",
        "performanceefficiency": "performance efficiency",
        "timebehaviour": "time behaviour",
        "resourceutilization": "resource utilization",
        "faulttolerance": "fault tolerance",
        "userinterfaceaesthetics": "user interface aesthetics",
        "usererrorprotection": "user error protection",
        "appropriatenessrecognizability": "appropriateness recognizability",
    }
    return replacements.get(norm_label(label_id), str(label_id))


def find_col(df: pd.DataFrame, candidates: List[str], required: bool = True) -> Optional[str]:
    lower_map = {c.lower().strip(): c for c in df.columns}
    for cand in candidates:
        key = cand.lower().strip()
        if key in lower_map:
            return lower_map[key]
    # relaxed: ignore spaces/underscores/case
    relaxed_map = {re.sub(r"[^a-z0-9]", "", c.lower()): c for c in df.columns}
    for cand in candidates:
        key = re.sub(r"[^a-z0-9]", "", cand.lower())
        if key in relaxed_map:
            return relaxed_map[key]
    if required:
        raise ValueError(f"Could not find any of these columns: {candidates}. Existing columns: {list(df.columns)}")
    return None


def softmax_np(x: np.ndarray, axis: int = -1) -> np.ndarray:
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)


seed_everything(cfg.SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [ ]:
# =========================
# 4) Load dataset and ISO cards
# =========================
def load_dataset(cfg: CFG) -> Tuple[pd.DataFrame, str, str, str]:
    """Load the prepared dataset unchanged and validate its saved folds."""
    path = cfg.CV_DATASET_PATH
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    if path.lower().endswith((".xlsx", ".xls")):
        data = pd.read_excel(path)
    elif path.lower().endswith(".csv"):
        data = pd.read_csv(path)
    else:
        raise ValueError(f"Unsupported dataset format: {path}")

    text_col = cfg.TEXT_COL or find_col(data, [
        "text", "requirement", "requirements", "Requirement", "sentence", "description", "RequirementText", "req_text"
    ])
    l1_col = cfg.L1_COL or find_col(data, ["L1", "l1", "parent", "Parent", "main_label", "Level1"])
    l2_col = cfg.L2_COL or find_col(data, ["L2", "l2", "label", "Label", "class", "Class", "Level2"])

    required = ["id", "orig_id", "source_split", "cv_index", text_col, l1_col, l2_col, "test_fold"]
    missing = [c for c in required if c not in data.columns]
    if missing:
        raise ValueError(
            f"Prepared CV dataset is missing required columns: {missing}. "
            "Use full_dataset_with_test_fold.xlsx from the preparation notebook."
        )
    if data[required].isna().any().any():
        bad_columns = data[required].columns[data[required].isna().any()].tolist()
        raise ValueError(f"Prepared CV dataset contains missing values in: {bad_columns}")

    data = data.copy()
    data["cv_index"] = pd.to_numeric(data["cv_index"], errors="raise").astype(int)
    data["test_fold"] = pd.to_numeric(data["test_fold"], errors="raise").astype(int)
    if data["cv_index"].duplicated().any():
        raise ValueError("cv_index is not unique in the prepared CV dataset.")
    if data["id"].astype(str).duplicated().any():
        raise ValueError("id is not unique in the prepared CV dataset.")

    expected_folds = set(range(cfg.N_FOLDS))
    observed_folds = set(data["test_fold"].unique().tolist())
    if observed_folds != expected_folds:
        raise ValueError(
            f"Expected saved folds {sorted(expected_folds)}, found {sorted(observed_folds)}."
        )

    # Compatibility alias used by the existing prediction-reporting code.
    data["global_row_id"] = data["cv_index"]

    # Do not repeat merging, cleaning, label normalization, filtering, or deduplication.
    data = data.sort_values("cv_index").reset_index(drop=True)

    print("Loaded dataset:", data.shape)
    print("Text column:", text_col, "L1 column:", l1_col, "L2 column:", l2_col)
    print("Number of L1:", data[l1_col].nunique(), "Number of L2:", data[l2_col].nunique())
    print("Saved test-fold sizes:")
    print(data["test_fold"].value_counts().sort_index())

    return data, text_col, l1_col, l2_col


def build_card(row: pd.Series, mode: str) -> str:
    l1_id = norm_label(row.get("L1", ""))
    l2_id = norm_label(row.get("L2", ""))

    l1_name = safe_cell(row.get("L1_name", "")) or pretty_label_from_id(l1_id)
    l2_name = safe_cell(row.get("L2_name", "")) or pretty_label_from_id(l2_id)
    iso_definition = safe_cell(row.get("iso_definition", ""))
    l1_definition = safe_cell(row.get("l1_definition", ""))
    signals = safe_cell(row.get("signals", ""))
    sibling = safe_cell(row.get("sibling_distinguish", ""))

    # Always include readable label names because this helps the encoder.
    prefix = (
        f"Quality sub-characteristic: {l2_name}. "
        f"Parent quality characteristic: {l1_name}. "
    )

    if mode == "signals_only":
        return prefix + f"Typical requirement signals: {signals}."

    if mode == "definition_only":
        return prefix + f"Definition: {iso_definition}."

    if mode == "definition_plus_signals":
        return prefix + f"Definition: {iso_definition}. Typical requirement signals: {signals}."

    if mode == "signals_plus_sibling":
        return prefix + f"Typical requirement signals: {signals}. Sibling distinction: {sibling}."

    if mode == "full":
        return (
            prefix
            + f"Definition: {iso_definition}. "
            + f"Parent definition: {l1_definition}. "
            + f"Typical requirement signals: {signals}. "
            + f"Sibling distinction: {sibling}."
        )

    raise ValueError(f"Unknown CARD_TEXT_MODE: {mode}")


def resolve_full_card_path(cfg: CFG) -> Optional[str]:
    """
    Resolve the improved full contrastive card file path.
    Returns None if no file is found.
    """
    import glob

    candidates = []
    if safe_cell(cfg.FULL_CARD_PATH):
        candidates.append(cfg.FULL_CARD_PATH)

    for fname in cfg.FULL_CARD_FALLBACK_FILENAMES:
        candidates.append(os.path.join("/content", fname))
        candidates.append(os.path.join("/content/drive/MyDrive", fname))

    # direct checks
    seen = set()
    for p in candidates:
        if p and p not in seen:
            seen.add(p)
            if os.path.exists(p):
                return p

    # limited recursive search
    for root in ["/content", "/content/drive/MyDrive"]:
        if os.path.exists(root):
            for fname in cfg.FULL_CARD_FALLBACK_FILENAMES:
                matches = glob.glob(os.path.join(root, "**", fname), recursive=True)
                if matches:
                    return matches[0]

    return None


def apply_external_full_contrastive_cards(
    iso: pd.DataFrame,
    cfg: CFG,
) -> pd.DataFrame:
    """
    Replace iso['label_card'] with cfg.FULL_CARD_COL from the improved workbook.

    In this experiment, iso['label_card'] is used only for Qwen3 retrieval.
    The DeBERTa reranker uses iso['reranker_label_card'], which is shorter.
    """
    if not cfg.USE_EXTERNAL_FULL_CONTRASTIVE_CARDS:
        iso["card_source"] = f"internal_{cfg.CARD_TEXT_MODE}"
        return iso

    full_path = resolve_full_card_path(cfg)
    if full_path is None:
        raise FileNotFoundError(
            "Could not find the improved full contrastive card file.\n"
            f"Configured FULL_CARD_PATH: {cfg.FULL_CARD_PATH}\n"
            "Upload improved_iso_label_cards_full_contrastive.xlsx to /content "
            "or update cfg.FULL_CARD_PATH to the exact Drive path."
        )

    print("=" * 90)
    print("Using external full contrastive cards from:", full_path)
    print("=" * 90)

    full_df = pd.read_excel(full_path, sheet_name=cfg.FULL_CARD_SHEET)

    l2_col = find_col(full_df, ["L2", "l2", "label", "Label", "L2_name", "l2_name"])
    l1_col = find_col(full_df, ["L1", "l1", "parent", "Parent", "L1_name", "l1_name"], required=False)

    if cfg.FULL_CARD_COL not in full_df.columns:
        raise ValueError(
            f"Could not find FULL_CARD_COL='{cfg.FULL_CARD_COL}'. "
            f"Available columns: {list(full_df.columns)}"
        )

    full_df = full_df.copy()
    full_df["L2_norm_for_map"] = full_df[l2_col].map(norm_label)
    full_df[cfg.FULL_CARD_COL] = full_df[cfg.FULL_CARD_COL].map(safe_cell)
    full_df = full_df[full_df[cfg.FULL_CARD_COL].str.len() > 0].drop_duplicates("L2_norm_for_map")

    full_card_map = dict(zip(full_df["L2_norm_for_map"], full_df[cfg.FULL_CARD_COL]))

    # Optional parent map from improved file, if present.
    external_parent_map = {}
    if l1_col is not None:
        external_parent_map = dict(zip(full_df["L2_norm_for_map"], full_df[l1_col].map(norm_label)))

    out = iso.copy()
    out["internal_label_card"] = out["label_card"]
    out["full_contrastive_label_card"] = out["L2"].map(lambda x: full_card_map.get(norm_label(x), ""))

    out["label_card"] = out.apply(
        lambda r: r["full_contrastive_label_card"]
        if safe_cell(r["full_contrastive_label_card"]) != ""
        else r["internal_label_card"],
        axis=1,
    )

    out["card_source"] = out["L2"].map(
        lambda x: "external_full_contrastive" if norm_label(x) in full_card_map else f"internal_{cfg.CARD_TEXT_MODE}"
    )

    # Fill/repair parent if original ISO parent is missing.
    if external_parent_map:
        out["L1"] = out.apply(
            lambda r: external_parent_map.get(norm_label(r["L2"]), r["L1"]) if safe_cell(r["L1"]) == "" else r["L1"],
            axis=1,
        )

    out["label_card_char_len"] = out["label_card"].astype(str).str.len()

    missing = out[out["card_source"] != "external_full_contrastive"]["L2"].tolist()
    if missing:
        print(f"WARNING: {len(missing)} L2 labels did not find external full card. Internal fallback used:")
        print(missing)

    # Strong sanity check: avoid silently using old cards.
    avg_len = float(out["label_card_char_len"].mean())
    preview = str(out["label_card"].iloc[0]) if len(out) else ""

    print("Final card sources:", out["card_source"].value_counts().to_dict())
    print("Average label card length:", round(avg_len, 1))
    print("Sanity preview:")
    print(preview[:800])

    if cfg.USE_EXTERNAL_FULL_CONTRASTIVE_CARDS:
        if "external_full_contrastive" not in set(out["card_source"].unique()):
            raise RuntimeError("External full contrastive cards were requested but no label used them.")
        if avg_len < 300:
            raise RuntimeError(
                "Average label card length is very short. This likely means old cards are still active."
            )

    return out



def extract_google_sheet_id(url: str) -> Optional[str]:
    m = re.search(r"/spreadsheets/d/([a-zA-Z0-9_-]+)", str(url))
    return m.group(1) if m else None


def read_excel_path_or_google_sheet(path_or_url: str, sheet_name: Optional[str] = None) -> pd.DataFrame:
    """
    Read an Excel/Google Sheet from:
    - local/Drive xlsx path, or
    - Google Sheets URL using export?format=xlsx.

    If the Google Sheet is private and Colab cannot access it, this may fail.
    In that case, download it as xlsx into Drive and set RERANKER_SHORT_CARD_PATH_OR_URL to the local path.
    """
    p = str(path_or_url).strip()
    if p == "":
        raise FileNotFoundError("Empty path_or_url")

    if p.startswith("http") and "docs.google.com/spreadsheets" in p:
        sid = extract_google_sheet_id(p)
        if sid is None:
            raise ValueError(f"Could not extract Google Sheet id from: {p}")
        export_url = f"https://docs.google.com/spreadsheets/d/{sid}/export?format=xlsx"
        return pd.read_excel(export_url, sheet_name=sheet_name if sheet_name else 0)

    return pd.read_excel(p, sheet_name=sheet_name if sheet_name else 0)


def apply_external_reranker_short_cards(
    iso: pd.DataFrame,
    cfg: CFG,
) -> pd.DataFrame:
    """
    Optional: override iso['reranker_label_card'] with the saved previous card file.
    If loading fails, the code falls back to the internal definition_plus_signals card.
    """
    if not getattr(cfg, "USE_EXTERNAL_RERANKER_SHORT_CARDS", False):
        return iso

    try:
        short_df = read_excel_path_or_google_sheet(
            cfg.RERANKER_SHORT_CARD_PATH_OR_URL,
            sheet_name=cfg.RERANKER_SHORT_CARD_SHEET,
        )
    except Exception as e:
        print("WARNING: Could not load external reranker short cards. Falling back to internal definition_plus_signals cards.")
        print("Reason:", repr(e))
        return iso

    l2_col = find_col(short_df, ["L2", "l2", "label", "Label", "L2_name", "l2_name"], required=False)
    if l2_col is None:
        print("WARNING: External reranker card file has no L2 column. Falling back to internal cards.")
        print("Available columns:", list(short_df.columns))
        return iso

    card_col = None
    for cand in cfg.RERANKER_SHORT_CARD_COL_CANDIDATES:
        if cand in short_df.columns:
            card_col = cand
            break

    if card_col is None:
        print("WARNING: External reranker card file has no recognized card column. Falling back to internal cards.")
        print("Available columns:", list(short_df.columns))
        return iso

    tmp = short_df.copy()
    tmp["L2_norm_for_map"] = tmp[l2_col].map(norm_label)
    tmp[card_col] = tmp[card_col].map(safe_cell)
    tmp = tmp[tmp[card_col].str.len() > 0].drop_duplicates("L2_norm_for_map")
    card_map = dict(zip(tmp["L2_norm_for_map"], tmp[card_col]))

    out = iso.copy()
    out["external_reranker_short_card"] = out["L2"].map(lambda x: card_map.get(norm_label(x), ""))
    out["reranker_label_card"] = out.apply(
        lambda r: r["external_reranker_short_card"]
        if safe_cell(r["external_reranker_short_card"]) != ""
        else r["reranker_label_card"],
        axis=1,
    )
    out["reranker_card_source"] = out.apply(
        lambda r: "external_previous_card"
        if safe_cell(r.get("external_reranker_short_card", "")) != ""
        else "internal_definition_plus_signals",
        axis=1,
    )

    print("Reranker short card sources:", out["reranker_card_source"].value_counts().to_dict())
    print("Reranker card average length:", round(out["reranker_label_card"].astype(str).str.len().mean(), 1))
    print("Reranker card preview:")
    print(str(out["reranker_label_card"].iloc[0])[:700])

    return out



def load_iso_cards(cfg: CFG, all_l2_from_data: List[str]) -> Tuple[pd.DataFrame, Dict[str, str], Dict[str, str], Dict[str, str], Dict[str, str]]:
    iso = pd.read_excel(cfg.ISO_CARD_PATH)
    required_cols = ["L1", "L2", "L1_name", "L2_name", "iso_definition", "l1_definition", "signals", "sibling_distinguish"]

    # Map columns robustly to exact names.
    renamed = {}
    for col in required_cols:
        found = find_col(iso, [col], required=(col in ["L1", "L2"]))
        if found is not None:
            renamed[found] = col
    iso = iso.rename(columns=renamed)

    for col in required_cols:
        if col not in iso.columns:
            iso[col] = ""

    iso = iso.copy()
    iso["L1"] = iso["L1"].map(norm_label)
    iso["L2"] = iso["L2"].map(norm_label)

    # Keep L2 rows only.
    iso = iso[(iso["L1"] != "") & (iso["L2"] != "")].copy()
    iso = iso.drop_duplicates(subset=["L2"], keep="first").reset_index(drop=True)
    # Short definition+signals card used by the DeBERTa reranker.
    iso["reranker_label_card"] = iso.apply(lambda r: build_card(r, cfg.CARD_TEXT_MODE), axis=1)
    iso["reranker_card_source"] = f"internal_{cfg.CARD_TEXT_MODE}"

    # label_card is the retrieval card. It starts as the short card, then is overridden by the full contrastive card.
    iso["label_card"] = iso["reranker_label_card"]
    iso["card_source"] = f"internal_{cfg.CARD_TEXT_MODE}"

    # Override label_card with full contrastive cards for Qwen3 retrieval only.
    iso = apply_external_full_contrastive_cards(iso, cfg)

    # Optional override for reranker short cards from your saved previous card workbook/sheet.
    iso = apply_external_reranker_short_cards(iso, cfg)

    retrieval_card_map = dict(zip(iso["L2"], iso["label_card"]))              # full card for Qwen3 retrieval
    reranker_card_map = dict(zip(iso["L2"], iso["reranker_label_card"]))      # short card for DeBERTa reranking
    label_to_parent = dict(zip(iso["L2"], iso["L1"]))
    label_name_map = dict(zip(iso["L2"], iso["L2_name"].map(safe_cell)))

    missing = sorted(set(all_l2_from_data) - set(retrieval_card_map.keys()))
    if missing:
        print("WARNING: These dataset L2 labels are missing from ISO card file:", missing)
        print("Fallback cards will be created using label IDs only.")
        for l2 in missing:
            retrieval_card_map[l2] = f"Quality sub-characteristic: {pretty_label_from_id(l2)}. Typical requirement signals: {pretty_label_from_id(l2)}."
            reranker_card_map[l2] = retrieval_card_map[l2]
            label_to_parent[l2] = "unknownparent"
            label_name_map[l2] = pretty_label_from_id(l2)

    # Save audit.
    audit_cols = [
        "L1", "L2", "L1_name", "L2_name", "iso_definition", "signals",
        "sibling_distinguish", "card_source", "label_card_char_len", "label_card", "reranker_card_source", "reranker_label_card"
    ]
    audit_cols = [c for c in audit_cols if c in iso.columns]
    audit = iso[audit_cols].copy()
    audit_path = os.path.join(cfg.OUT_DIR, "label_cards_audit_QWEN3_FULL_RETRIEVAL_DEBERTA_SHORT_RERANKER.xlsx")
    audit.to_excel(audit_path, index=False)
    print("Saved label card audit:", audit_path)

    return iso, retrieval_card_map, reranker_card_map, label_to_parent, label_name_map


In [ ]:
# =========================
# 5) Embeddings and retrieval
# =========================
def l2_normalize(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    return x / np.maximum(np.linalg.norm(x, axis=1, keepdims=True), eps)


def is_qwen3_embedding_model(model_name: str) -> bool:
    return "qwen3-embedding" in str(model_name).lower()


def qwen3_prompt(is_query: bool, text_type: str = "document") -> str:
    """
    Qwen3 supports instruction-aware query embeddings.
    We use instructions for requirement queries, and keep label/training corpus texts unprompted by default.
    """
    if not cfg.USE_QWEN3_INSTRUCTION:
        return ""

    if is_query:
        return f"Instruct: {cfg.QWEN3_QUERY_INSTRUCTION}\nQuery: "

    if safe_cell(cfg.QWEN3_DOC_INSTRUCTION) != "":
        return f"Instruct: {cfg.QWEN3_DOC_INSTRUCTION}\nDocument: "

    return ""


def encode_texts(
    model: SentenceTransformer,
    texts: List[str],
    batch_size: int = 64,
    is_query: bool = False,
    text_type: str = "document",
) -> np.ndarray:
    """
    Return L2-normalized embeddings.

    For Qwen3:
    - val/test requirement texts are query embeddings with instruction;
    - training requirement corpus and label cards are document embeddings.
    """
    texts = [str(t) for t in texts]

    prompt = ""
    if is_qwen3_embedding_model(cfg.RETRIEVER_MODEL_NAME):
        prompt = qwen3_prompt(is_query=is_query, text_type=text_type)

    kwargs = dict(
        sentences=texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    # Newer sentence-transformers supports prompt=...
    # Older versions may not; then we manually prepend the prompt.
    try:
        if prompt:
            emb = model.encode(prompt=prompt, **kwargs)
        else:
            emb = model.encode(**kwargs)
    except TypeError:
        if prompt:
            kwargs["sentences"] = [prompt + t for t in texts]
        emb = model.encode(**kwargs)

    return np.asarray(emb, dtype=np.float32)


def aggregate_knn_scores(
    neighbor_labels: List[str],
    neighbor_scores: List[float],
    mode: str = "max",
) -> Dict[str, float]:
    d: Dict[str, List[float]] = {}
    for lab, sc in zip(neighbor_labels, neighbor_scores):
        d.setdefault(lab, []).append(float(sc))

    out = {}
    for lab, vals in d.items():
        if mode == "max":
            out[lab] = max(vals)
        elif mode == "mean":
            out[lab] = float(np.mean(vals))
        elif mode == "sum":
            out[lab] = float(np.sum(vals))
        else:
            raise ValueError(f"Unknown KNN_SCORE_MODE: {mode}")
    return out


def build_candidates_for_split(
    query_emb: np.ndarray,
    train_emb: np.ndarray,
    label_emb: np.ndarray,
    label_ids: List[str],
    train_l2: List[str],
    mode: str,
    candidate_k: int,
    knn_top_n: int,
    iso_weight: float,
    knn_weight: float,
    knn_score_mode: str,
    query_train_positions: Optional[List[Optional[int]]] = None,
) -> Tuple[Dict[int, List[str]], pd.DataFrame]:
    """
    Return candidate_map where candidate_map[i] is a list of candidate L2 IDs for query row i.

    query_train_positions:
      - For train queries, pass their position in train_emb to exclude self-neighbor.
      - For val/test queries, pass None.
    """
    n_query = query_emb.shape[0]
    n_labels = len(label_ids)

    # No-retrieval ablation: expose the complete label set to the reranker.
    if mode == "full_label":
        candidate_map = {i: list(label_ids) for i in range(n_query)}
        rows = []
        for i in range(n_query):
            row = {"row_pos": i, "candidate_at_full": "|".join(label_ids)}
            for k in cfg.RETRIEVAL_RECALL_KS:
                row[f"candidate_at_{k}"] = "|".join(label_ids[:min(k, n_labels)])
            rows.append(row)
        return candidate_map, pd.DataFrame(rows)

    # ISO label-card similarity
    iso_sims = query_emb @ label_emb.T  # [N, num_labels]

    # kNN training-example similarity
    use_knn = mode in {"knn_only", "iso_plus_knn"}
    if use_knn:
        sims_train = query_emb @ train_emb.T
        if query_train_positions is not None:
            for qi, train_pos in enumerate(query_train_positions):
                if train_pos is not None and 0 <= train_pos < sims_train.shape[1]:
                    sims_train[qi, train_pos] = -999.0
    else:
        sims_train = None

    candidate_map: Dict[int, List[str]] = {}
    rows = []

    for i in range(n_query):
        score_map: Dict[str, float] = {}

        if mode in {"iso_only", "iso_plus_knn"}:
            for j, lab in enumerate(label_ids):
                score_map[lab] = score_map.get(lab, 0.0) + iso_weight * float(iso_sims[i, j])

        if mode in {"knn_only", "iso_plus_knn"}:
            if sims_train is None:
                raise ValueError("sims_train is None but kNN mode was requested.")
            top_n = min(knn_top_n, sims_train.shape[1])
            top_idx = np.argpartition(-sims_train[i], range(top_n))[:top_n]
            top_idx = top_idx[np.argsort(-sims_train[i, top_idx])]

            neigh_labels = [train_l2[t] for t in top_idx]
            neigh_scores = [float(sims_train[i, t]) for t in top_idx]
            knn_scores = aggregate_knn_scores(neigh_labels, neigh_scores, mode=knn_score_mode)

            for lab, sc in knn_scores.items():
                score_map[lab] = score_map.get(lab, 0.0) + knn_weight * sc

        # Fallback
        if not score_map:
            score_map = {lab: 0.0 for lab in label_ids}

        ranked = sorted(score_map.items(), key=lambda x: x[1], reverse=True)
        candidates = [lab for lab, _ in ranked[:candidate_k]]
        candidate_map[i] = candidates

        row = {"row_pos": i}
        for k in cfg.RETRIEVAL_RECALL_KS:
            row[f"candidate_at_{k}"] = "|".join([lab for lab, _ in ranked[:k]])
        rows.append(row)

    return candidate_map, pd.DataFrame(rows)


def retrieval_recall_metrics(
    df_split: pd.DataFrame,
    candidate_map_full_rank: Dict[int, List[str]],
    l2_col: str,
    ks: Tuple[int, ...],
) -> Dict[str, float]:
    gold = df_split[l2_col].tolist()
    out = {}
    for k in ks:
        ok = []
        for i, y in enumerate(gold):
            ok.append(y in candidate_map_full_rank[i][:k])
        out[f"recall_at_{k}"] = float(np.mean(ok))
        out[f"gold_missing_at_{k}"] = 1.0 - out[f"recall_at_{k}"]
    return out

In [ ]:
# =========================
# 6) Listwise dataset and loss
# =========================
class ListwiseRerankDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        candidate_map: Dict[int, List[str]],
        label_card_map: Dict[str, str],
        tokenizer: AutoTokenizer,
        text_col: str,
        l2_col: str,
        candidate_k: int = 10,
        max_len: int = 256,
        force_gold: bool = True,
        all_l2_labels: Optional[List[str]] = None,
        label_to_parent: Optional[Dict[str, str]] = None,
        use_hard_sibling_negatives: bool = False,
        n_hard_sibling_negatives: int = 3,
        hard_sibling_strategy: str = "mixed",
        random_seed: int = 42,
    ):
        self.df = df.reset_index(drop=True)
        self.candidate_map = candidate_map
        self.label_card_map = label_card_map
        self.tokenizer = tokenizer
        self.text_col = text_col
        self.l2_col = l2_col
        self.candidate_k = candidate_k
        self.max_len = max_len
        self.force_gold = force_gold
        self.all_l2_labels = all_l2_labels or sorted(label_card_map.keys())

        # Hard sibling negatives are used only when force_gold=True, i.e., training.
        self.label_to_parent = label_to_parent or {}
        self.use_hard_sibling_negatives = bool(use_hard_sibling_negatives and force_gold)
        self.n_hard_sibling_negatives = int(n_hard_sibling_negatives)
        self.hard_sibling_strategy = hard_sibling_strategy
        self.random_seed = int(random_seed)

        self.parent_to_labels: Dict[str, List[str]] = {}
        for lab in self.all_l2_labels:
            parent = self.label_to_parent.get(lab, "unknown")
            self.parent_to_labels.setdefault(parent, []).append(lab)
        for parent in self.parent_to_labels:
            self.parent_to_labels[parent] = sorted(set(self.parent_to_labels[parent]))

    def __len__(self):
        return len(self.df)

    def _select_hard_siblings(self, gold_l2: str, candidates: List[str], idx: int) -> List[str]:
        """
        Select hard negatives from the same ISO L1 parent as the gold L2.

        mixed strategy:
          1) keep sibling labels that already appear in the retriever candidate list;
          2) fill the remaining slots by random sibling labels from the same parent.
        This keeps the training deterministic using random_seed + idx.
        """
        if not self.use_hard_sibling_negatives or self.n_hard_sibling_negatives <= 0:
            return []

        parent = self.label_to_parent.get(gold_l2, None)
        if parent is None:
            return []

        sibling_pool = [
            lab for lab in self.parent_to_labels.get(parent, [])
            if lab != gold_l2
        ]
        if not sibling_pool:
            return []

        # Siblings already retrieved are usually the hardest because the retriever considered them relevant.
        retrieved_siblings = [c for c in candidates if c in sibling_pool]

        # Fill the remaining slots with other siblings from the same parent.
        missing_siblings = [s for s in sibling_pool if s not in retrieved_siblings]
        rng = random.Random(self.random_seed + idx)
        rng.shuffle(missing_siblings)

        if self.hard_sibling_strategy == "retrieved_first":
            hard_sibs = retrieved_siblings + missing_siblings
        elif self.hard_sibling_strategy == "random":
            all_sibs = sibling_pool[:]
            rng.shuffle(all_sibs)
            hard_sibs = all_sibs
        elif self.hard_sibling_strategy == "mixed":
            hard_sibs = retrieved_siblings + missing_siblings
        else:
            raise ValueError(f"Unknown HARD_SIBLING_STRATEGY: {self.hard_sibling_strategy}")

        return hard_sibs[: self.n_hard_sibling_negatives]

    def _force_protected_candidates(
        self,
        candidates: List[str],
        protected: List[str],
    ) -> List[str]:
        """
        Add protected labels, then trim from the end while never removing protected labels.
        Protected labels are usually: [gold_l2] + hard sibling negatives.
        """
        # Remove duplicates while preserving order.
        seen = set()
        candidates = [c for c in candidates if not (c in seen or seen.add(c))]

        for lab in protected:
            if lab and lab not in candidates:
                candidates.append(lab)

        protected_set = set([p for p in protected if p])

        # If too long, remove lowest-priority non-protected candidates from the end.
        while len(candidates) > self.candidate_k:
            remove_idx = None
            for j in range(len(candidates) - 1, -1, -1):
                if candidates[j] not in protected_set:
                    remove_idx = j
                    break

            # This should only happen if candidate_k is smaller than number of protected labels.
            if remove_idx is None:
                # Always keep gold first, then as many hard siblings as fit.
                candidates = protected[: self.candidate_k]
                break

            candidates.pop(remove_idx)

        return candidates

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        req_text = clean_text(row[self.text_col])
        gold_l2 = norm_label(row[self.l2_col])

        candidates = list(self.candidate_map.get(idx, []))[:self.candidate_k]

        # Fallback if candidate list is empty.
        if not candidates:
            candidates = self.all_l2_labels[:self.candidate_k]

        # Training only:
        # 1) force the gold label into the list;
        # 2) optionally force hard sibling negatives from the same ISO L1 parent.
        hard_siblings: List[str] = []
        if self.force_gold:
            if self.use_hard_sibling_negatives:
                hard_siblings = self._select_hard_siblings(gold_l2, candidates, idx)
            candidates = self._force_protected_candidates(
                candidates=candidates,
                protected=[gold_l2] + hard_siblings,
            )
        else:
            # Evaluation: do NOT force gold or extra siblings, to keep retrieval realistic.
            seen = set()
            candidates = [c for c in candidates if not (c in seen or seen.add(c))]

        # Pad to candidate_k. Padding with the last candidate is okay; it only repeats an existing option.
        while len(candidates) < self.candidate_k:
            if candidates:
                candidates.append(candidates[-1])
            else:
                candidates.append(self.all_l2_labels[0])

        candidates = candidates[:self.candidate_k]

        gold_index = candidates.index(gold_l2) if gold_l2 in candidates else -1
        label_cards = [self.label_card_map.get(c, pretty_label_from_id(c)) for c in candidates]

        enc = self.tokenizer(
            [req_text] * self.candidate_k,
            label_cards,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )

        item = {
            "input_ids": enc["input_ids"],              # [K, L]
            "attention_mask": enc["attention_mask"],    # [K, L]
            "gold_index": torch.tensor(gold_index, dtype=torch.long),
            "gold_l2": gold_l2,
            "candidate_l2": candidates,
            "hard_sibling_l2": hard_siblings,
        }
        if "token_type_ids" in enc:
            item["token_type_ids"] = enc["token_type_ids"]
        return item


def listwise_collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    out = {
        "input_ids": torch.stack([x["input_ids"] for x in batch]),
        "attention_mask": torch.stack([x["attention_mask"] for x in batch]),
        "gold_index": torch.stack([x["gold_index"] for x in batch]),
        "gold_l2": [x["gold_l2"] for x in batch],
        "candidate_l2": [x["candidate_l2"] for x in batch],
        "hard_sibling_l2": [x.get("hard_sibling_l2", []) for x in batch],
    }
    if "token_type_ids" in batch[0]:
        out["token_type_ids"] = torch.stack([x["token_type_ids"] for x in batch])
    return out


def make_class_weight_map(train_df: pd.DataFrame, l2_col: str, all_l2_labels: List[str], clip: float = 5.0) -> Dict[str, float]:
    counts = train_df[l2_col].value_counts().to_dict()
    total = len(train_df)
    n_classes = len(all_l2_labels)
    weights = {}
    for lab in all_l2_labels:
        c = counts.get(lab, 0)
        if c <= 0:
            w = 1.0
        else:
            w = total / (n_classes * c)
        weights[lab] = float(np.clip(w, 1.0 / clip, clip))
    return weights


def compute_l1_aux_loss_from_candidate_scores(
    scores: torch.Tensor,
    batch: Dict[str, Any],
    label_to_parent: Dict[str, str],
    device: str,
    pool: str = "logmeanexp",
) -> Optional[torch.Tensor]:
    """
    Auxiliary L1/parent CE computed from the same listwise candidate scores.

    For each requirement, candidate L2 scores [K] are aggregated into parent/L1 logits.
    This encourages the reranker to prefer candidates under the correct ISO parent, while
    the main listwise loss still chooses the exact L2 label.

    Duplicate padded candidates are ignored so that padding does not inflate any parent logit.
    """
    parent_losses = []
    B, _ = scores.shape

    for i in range(B):
        gold_l2 = batch["gold_l2"][i]
        gold_parent = label_to_parent.get(gold_l2, "unknown")
        if gold_parent == "unknown":
            continue

        candidates = batch["candidate_l2"][i]

        parent_to_positions: Dict[str, List[int]] = {}
        seen_l2 = set()
        for j, lab in enumerate(candidates):
            # Ignore duplicated labels introduced by padding/repetition.
            if lab in seen_l2:
                continue
            seen_l2.add(lab)
            parent = label_to_parent.get(lab, "unknown")
            parent_to_positions.setdefault(parent, []).append(j)

        if gold_parent not in parent_to_positions:
            # This should not happen during training because gold L2 is forced into candidates.
            continue

        parent_order = sorted(parent_to_positions.keys())
        parent_logits = []
        for parent in parent_order:
            idxs = torch.tensor(parent_to_positions[parent], dtype=torch.long, device=device)
            vals = scores[i].index_select(0, idxs)

            if pool == "max":
                parent_logit = vals.max()
            elif pool == "mean":
                parent_logit = vals.mean()
            elif pool == "logsumexp":
                parent_logit = torch.logsumexp(vals, dim=0)
            elif pool == "logmeanexp":
                # Safer than plain logsumexp because parents with more candidate labels
                # should not win only because they have more candidates.
                parent_logit = torch.logsumexp(vals, dim=0) - math.log(max(len(parent_to_positions[parent]), 1))
            else:
                raise ValueError(f"Unknown L1_AUX_POOL: {pool}")

            parent_logits.append(parent_logit)

        target_idx = parent_order.index(gold_parent)
        parent_logits_t = torch.stack(parent_logits).unsqueeze(0)  # [1, num_candidate_parents]
        target_t = torch.tensor([target_idx], dtype=torch.long, device=device)
        parent_losses.append(F.cross_entropy(parent_logits_t, target_t))

    if not parent_losses:
        return None

    return torch.stack(parent_losses).mean()



def compute_hierarchy_margin_loss_from_candidate_scores(
    scores: torch.Tensor,
    batch: Dict[str, Any],
    label_to_parent: Dict[str, str],
    device: str,
    margin_sibling: float = 0.20,
    margin_global: float = 0.05,
    include_global_negatives: bool = True,
) -> Optional[torch.Tensor]:
    """
    Hierarchy-aware margin ranking loss over candidate L2 scores.

    For each valid requirement, the gold candidate score should exceed every negative
    candidate score by at least a margin. Same-parent/sibling negatives receive a
    larger margin than cross-parent negatives:

        loss_j = max(0, margin_j - s_gold + s_negative)

    where margin_j = margin_sibling if Parent(negative) == Parent(gold), otherwise
    margin_global. Duplicate candidates introduced by padding are ignored.

    This loss is useful when the model reaches the correct L1 parent but confuses
    sibling L2 labels. It does not change inference; prediction is still argmax over
    candidate L2 scores.
    """
    losses = []
    B, K = scores.shape
    gold_index = batch["gold_index"]

    # gold_index may be CPU tensor from the collate function. Keep indexing safe.
    if isinstance(gold_index, torch.Tensor):
        gold_index_cpu = gold_index.detach().cpu().tolist()
    else:
        gold_index_cpu = list(gold_index)

    for i in range(B):
        gi = int(gold_index_cpu[i])
        if gi < 0 or gi >= K:
            continue

        gold_l2 = batch["gold_l2"][i]
        gold_parent = label_to_parent.get(gold_l2, "unknown")
        if gold_parent == "unknown":
            continue

        candidates = batch["candidate_l2"][i]
        s_gold = scores[i, gi]

        item_losses = []
        seen_negatives = set()
        for j, neg_l2 in enumerate(candidates):
            if j == gi:
                continue
            if neg_l2 == gold_l2:
                # Ignore duplicate gold label if it appears due to padding or construction.
                continue
            if neg_l2 in seen_negatives:
                # Ignore duplicate padded/repeated negatives so they do not over-weight a label.
                continue
            seen_negatives.add(neg_l2)

            neg_parent = label_to_parent.get(neg_l2, "unknown")
            is_sibling = (neg_parent == gold_parent) and (neg_parent != "unknown")

            if is_sibling:
                margin = float(margin_sibling)
            else:
                if not include_global_negatives:
                    continue
                margin = float(margin_global)

            s_neg = scores[i, j]
            item_losses.append(F.relu(torch.tensor(margin, dtype=scores.dtype, device=device) - s_gold + s_neg))

        if item_losses:
            losses.append(torch.stack(item_losses).mean())

    if not losses:
        return None

    return torch.stack(losses).mean()


def compute_listwise_loss(
    model: AutoModelForSequenceClassification,
    batch: Dict[str, Any],
    device: str,
    class_weight_map: Optional[Dict[str, float]] = None,
    label_to_parent: Optional[Dict[str, str]] = None,
    lambda_l1_aux: float = 0.0,
    l1_aux_pool: str = "logmeanexp",
    lambda_sibling_margin: float = 0.0,
    margin_sibling: float = 0.20,
    margin_global: float = 0.05,
    sibling_margin_include_global_negatives: bool = True,
) -> Tuple[Optional[torch.Tensor], torch.Tensor, Dict[str, float]]:
    input_ids = batch["input_ids"].to(device)              # [B, K, L]
    attention_mask = batch["attention_mask"].to(device)    # [B, K, L]
    gold_index = batch["gold_index"].to(device)            # [B]

    B, K, L = input_ids.shape

    inputs = {
        "input_ids": input_ids.reshape(B * K, L),
        "attention_mask": attention_mask.reshape(B * K, L),
    }
    if "token_type_ids" in batch:
        inputs["token_type_ids"] = batch["token_type_ids"].to(device).reshape(B * K, L)

    outputs = model(**inputs)
    scores = outputs.logits.reshape(B, K)  # one score per candidate

    valid = gold_index >= 0
    if valid.sum().item() == 0:
        return None, scores, {
            "l2_loss": 0.0,
            "l1_aux_loss": 0.0,
            "sibling_margin_loss": 0.0,
            "total_loss": 0.0,
        }

    per_item_loss = F.cross_entropy(scores[valid], gold_index[valid], reduction="none")

    # Optional class weighting by gold L2.
    if class_weight_map is not None:
        weights = []
        valid_cpu = valid.detach().cpu().numpy().tolist()
        for i, is_valid in enumerate(valid_cpu):
            if is_valid:
                weights.append(class_weight_map.get(batch["gold_l2"][i], 1.0))
        weights_t = torch.tensor(weights, dtype=torch.float32, device=device)
        per_item_loss = per_item_loss * weights_t

    l2_loss = per_item_loss.mean()
    l1_aux_loss = None
    sibling_margin_loss = None

    if label_to_parent is not None and lambda_l1_aux > 0:
        l1_aux_loss = compute_l1_aux_loss_from_candidate_scores(
            scores=scores,
            batch=batch,
            label_to_parent=label_to_parent,
            device=device,
            pool=l1_aux_pool,
        )

    if label_to_parent is not None and lambda_sibling_margin > 0:
        sibling_margin_loss = compute_hierarchy_margin_loss_from_candidate_scores(
            scores=scores,
            batch=batch,
            label_to_parent=label_to_parent,
            device=device,
            margin_sibling=margin_sibling,
            margin_global=margin_global,
            include_global_negatives=sibling_margin_include_global_negatives,
        )

    total_loss = l2_loss

    if l1_aux_loss is not None:
        total_loss = total_loss + float(lambda_l1_aux) * l1_aux_loss
        l1_aux_loss_value = float(l1_aux_loss.detach().item())
    else:
        l1_aux_loss_value = 0.0

    if sibling_margin_loss is not None:
        total_loss = total_loss + float(lambda_sibling_margin) * sibling_margin_loss
        sibling_margin_loss_value = float(sibling_margin_loss.detach().item())
    else:
        sibling_margin_loss_value = 0.0

    loss_parts = {
        "l2_loss": float(l2_loss.detach().item()),
        "l1_aux_loss": l1_aux_loss_value,
        "sibling_margin_loss": sibling_margin_loss_value,
        "total_loss": float(total_loss.detach().item()),
    }
    return total_loss, scores, loss_parts


In [ ]:
# =========================
# 7) Metrics
# =========================
def compute_hierarchical_metrics(
    y_true_l2: List[str],
    y_pred_l2: List[str],
    label_to_parent: Dict[str, str],
) -> Dict[str, float]:
    hp_list, hr_list, hf_list = [], [], []
    correct_l2 = 0
    correct_l1_wrong_l2 = 0
    wrong_l1 = 0

    for yt, yp in zip(y_true_l2, y_pred_l2):
        true_parent = label_to_parent.get(yt, "unknown")
        pred_parent = label_to_parent.get(yp, "unknown")

        true_set = {true_parent, yt}
        pred_set = {pred_parent, yp}
        inter = len(true_set.intersection(pred_set))

        hp = inter / max(len(pred_set), 1)
        hr = inter / max(len(true_set), 1)
        hf = 0.0 if (hp + hr) == 0 else 2 * hp * hr / (hp + hr)

        hp_list.append(hp)
        hr_list.append(hr)
        hf_list.append(hf)

        if yt == yp:
            correct_l2 += 1
        elif true_parent == pred_parent:
            correct_l1_wrong_l2 += 1
        else:
            wrong_l1 += 1

    n = len(y_true_l2)
    return {
        "hP": float(np.mean(hp_list)),
        "hR": float(np.mean(hr_list)),
        "hF1": float(np.mean(hf_list)),
        "correct_l2_pct": correct_l2 / max(n, 1),
        "correct_l1_wrong_l2_pct": correct_l1_wrong_l2 / max(n, 1),
        "wrong_l1_pct": wrong_l1 / max(n, 1),
    }


def compute_final_metrics(
    y_true: List[str],
    y_pred: List[str],
    all_l2_labels: List[str],
    label_to_parent: Dict[str, str],
    prefix: str = "",
) -> Dict[str, float]:
    out = {
        f"{prefix}l2_macro_f1": f1_score(y_true, y_pred, labels=all_l2_labels, average="macro", zero_division=0),
        f"{prefix}l2_weighted_f1": f1_score(y_true, y_pred, labels=all_l2_labels, average="weighted", zero_division=0),
        f"{prefix}l2_accuracy": accuracy_score(y_true, y_pred),
        f"{prefix}l2_balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
    }

    true_parent = [label_to_parent.get(x, "unknown") for x in y_true]
    pred_parent = [label_to_parent.get(x, "unknown") for x in y_pred]
    parent_labels = sorted(set(true_parent) | set(pred_parent))
    out.update({
        f"{prefix}parent_macro_f1": f1_score(true_parent, pred_parent, labels=parent_labels, average="macro", zero_division=0),
        f"{prefix}parent_weighted_f1": f1_score(true_parent, pred_parent, labels=parent_labels, average="weighted", zero_division=0),
        f"{prefix}parent_accuracy": accuracy_score(true_parent, pred_parent),
    })

    h = compute_hierarchical_metrics(y_true, y_pred, label_to_parent)
    out.update({f"{prefix}{k}": v for k, v in h.items()})
    return out


@torch.no_grad()
def evaluate_listwise(
    model: AutoModelForSequenceClassification,
    dataloader: DataLoader,
    device: str,
    all_l2_labels: List[str],
    label_to_parent: Dict[str, str],
) -> Dict[str, Any]:
    model.eval()
    y_true, y_pred = [], []
    pred_scores = []
    pred_candidates = []

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        B, K, L = input_ids.shape

        inputs = {
            "input_ids": input_ids.reshape(B * K, L),
            "attention_mask": attention_mask.reshape(B * K, L),
        }
        if "token_type_ids" in batch:
            inputs["token_type_ids"] = batch["token_type_ids"].to(device).reshape(B * K, L)

        outputs = model(**inputs)
        scores = outputs.logits.reshape(B, K)
        probs = torch.softmax(scores, dim=1)
        best_idx = torch.argmax(scores, dim=1).detach().cpu().numpy().tolist()
        best_score = torch.max(probs, dim=1).values.detach().cpu().numpy().tolist()

        for i, bi in enumerate(best_idx):
            gold_l2 = batch["gold_l2"][i]
            pred_l2 = batch["candidate_l2"][i][bi]
            y_true.append(gold_l2)
            y_pred.append(pred_l2)
            pred_scores.append(float(best_score[i]))
            pred_candidates.append("|".join(batch["candidate_l2"][i]))

    metrics = compute_final_metrics(y_true, y_pred, all_l2_labels, label_to_parent)
    metrics["y_true"] = y_true
    metrics["y_pred"] = y_pred
    metrics["pred_score"] = pred_scores
    metrics["pred_candidates"] = pred_candidates
    return metrics

In [ ]:
# =========================
# 8) Model creation and training loop
# =========================
def create_reranker_model(cfg: CFG) -> AutoModelForSequenceClassification:
    config = AutoConfig.from_pretrained(cfg.RERANKER_MODEL_NAME, num_labels=1)

    # Dropout regularization.
    if hasattr(config, "hidden_dropout_prob"):
        config.hidden_dropout_prob = cfg.HIDDEN_DROPOUT
    if hasattr(config, "attention_probs_dropout_prob"):
        config.attention_probs_dropout_prob = cfg.ATTENTION_DROPOUT
    if hasattr(config, "classifier_dropout"):
        config.classifier_dropout = cfg.CLASSIFIER_DROPOUT
    if hasattr(config, "summary_first_dropout"):
        config.summary_first_dropout = cfg.CLASSIFIER_DROPOUT

    model = AutoModelForSequenceClassification.from_pretrained(
        cfg.RERANKER_MODEL_NAME,
        config=config,
    )

    # IMPORTANT:
    # Keep the trainable reranker in FP32.
    # Qwen3 retrieval can use FP16, but the reranker parameters should not be loaded as FP16
    # when using GradScaler, otherwise PyTorch raises:
    # ValueError: Attempting to unscale FP16 gradients.
    model = model.float()

    first_dtype = next(model.parameters()).dtype
    print("Reranker loaded:", cfg.RERANKER_MODEL_NAME)
    print("Reranker parameter dtype:", first_dtype)
    print("Model dtype:", next(model.parameters()).dtype)
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())
    print("Trainable params:", trainable_params)
    print("Total params:", total_params)
    print("Trainable ratio:", trainable_params / total_params)

    return model


def train_listwise_model(
    model: AutoModelForSequenceClassification,
    train_dataset: ListwiseRerankDataset,
    val_dataset: ListwiseRerankDataset,
    tokenizer: AutoTokenizer,
    fold_out_dir: str,
    all_l2_labels: List[str],
    label_to_parent: Dict[str, str],
    class_weight_map: Optional[Dict[str, float]],
    cfg: CFG,
) -> Tuple[AutoModelForSequenceClassification, Dict[str, Any]]:
    model.to(device)

    # Safety: ensure trainable reranker parameters are FP32.
    # This prevents GradScaler from seeing FP16 gradients.
    if next(model.parameters()).dtype == torch.float16:
        print("WARNING: Reranker parameters are FP16. Converting trainable model to FP32.")
        model.float()

    amp_enabled = bool(cfg.FP16 and device == "cuda")
    print("Reranker AMP enabled:", amp_enabled)
    print("Reranker dtype before training:", next(model.parameters()).dtype)

    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.TRAIN_BATCH_SIZE,
        shuffle=True,
        collate_fn=listwise_collate_fn,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.EVAL_BATCH_SIZE,
        shuffle=False,
        collate_fn=listwise_collate_fn,
    )

    optimizer = AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    total_steps = max(len(train_loader) * cfg.EPOCHS, 1)
    warmup_steps = int(total_steps * cfg.WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

    best_val_macro_f1 = -1.0
    best_state = None
    best_epoch = -1
    bad_epochs = 0
    history = []

    Path(fold_out_dir).mkdir(parents=True, exist_ok=True)

    for epoch in range(1, cfg.EPOCHS + 1):
        model.train()
        total_loss = 0.0
        total_l2_loss = 0.0
        total_l1_aux_loss = 0.0
        total_sibling_margin_loss = 0.0
        n_steps = 0

        for batch in tqdm(train_loader, desc=f"Epoch {epoch}"):
            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=amp_enabled):
                loss, _, loss_parts = compute_listwise_loss(
                    model=model,
                    batch=batch,
                    device=device,
                    class_weight_map=class_weight_map,
                    label_to_parent=label_to_parent,
                    lambda_l1_aux=(cfg.LAMBDA_L1_AUX if cfg.USE_L1_AUX else 0.0),
                    l1_aux_pool=cfg.L1_AUX_POOL,
                    lambda_sibling_margin=(cfg.LAMBDA_SIBLING_MARGIN if cfg.USE_SIBLING_MARGIN_LOSS else 0.0),
                    margin_sibling=cfg.MARGIN_SIBLING,
                    margin_global=cfg.MARGIN_GLOBAL,
                    sibling_margin_include_global_negatives=cfg.SIBLING_MARGIN_INCLUDE_GLOBAL_NEGATIVES,
                )

            if loss is None:
                continue

            if amp_enabled:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.MAX_GRAD_NORM)
                optimizer.step()

            scheduler.step()

            total_loss += float(loss.item())
            total_l2_loss += float(loss_parts.get("l2_loss", 0.0))
            total_l1_aux_loss += float(loss_parts.get("l1_aux_loss", 0.0))
            total_sibling_margin_loss += float(loss_parts.get("sibling_margin_loss", 0.0))
            n_steps += 1

        train_loss = total_loss / max(n_steps, 1)
        train_l2_loss = total_l2_loss / max(n_steps, 1)
        train_l1_aux_loss = total_l1_aux_loss / max(n_steps, 1)
        train_sibling_margin_loss = total_sibling_margin_loss / max(n_steps, 1)

        val_metrics = evaluate_listwise(
            model=model,
            dataloader=val_loader,
            device=device,
            all_l2_labels=all_l2_labels,
            label_to_parent=label_to_parent,
        )
        val_macro_f1 = val_metrics["l2_macro_f1"]

        epoch_row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_l2_loss": train_l2_loss,
            "train_l1_aux_loss": train_l1_aux_loss,
            "lambda_l1_aux": cfg.LAMBDA_L1_AUX if cfg.USE_L1_AUX else 0.0,
            "train_sibling_margin_loss": train_sibling_margin_loss,
            "lambda_sibling_margin": cfg.LAMBDA_SIBLING_MARGIN if cfg.USE_SIBLING_MARGIN_LOSS else 0.0,
            "margin_sibling": cfg.MARGIN_SIBLING,
            "margin_global": cfg.MARGIN_GLOBAL,
            "sibling_margin_include_global_negatives": cfg.SIBLING_MARGIN_INCLUDE_GLOBAL_NEGATIVES,
            "val_l2_macro_f1": val_metrics["l2_macro_f1"],
            "val_l2_weighted_f1": val_metrics["l2_weighted_f1"],
            "val_l2_accuracy": val_metrics["l2_accuracy"],
            "val_l2_balanced_accuracy": val_metrics["l2_balanced_accuracy"],
            "val_parent_macro_f1": val_metrics["parent_macro_f1"],
            "val_hF1": val_metrics["hF1"],
        }
        history.append(epoch_row)

        print(
            f"Epoch {epoch} | "
            f"loss={train_loss:.4f} "
            f"(l2={train_l2_loss:.4f}, l1aux={train_l1_aux_loss:.4f}, sibmargin={train_sibling_margin_loss:.4f}) | "
            f"val_l2_macro_f1={val_metrics['l2_macro_f1']:.4f} | "
            f"val_l2_bal_acc={val_metrics['l2_balanced_accuracy']:.4f} | "
            f"val_hF1={val_metrics['hF1']:.4f}"
        )

        # Model selection by validation final L2 macro-F1 after re-ranking.
        if val_macro_f1 > best_val_macro_f1:
            best_val_macro_f1 = val_macro_f1
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0

            if cfg.SAVE_BEST_MODEL:
                model.save_pretrained(fold_out_dir)
                tokenizer.save_pretrained(fold_out_dir)

            print(f"Saved best model at epoch {epoch}: val_l2_macro_f1={best_val_macro_f1:.4f}")
        else:
            bad_epochs += 1
            if bad_epochs >= cfg.PATIENCE:
                print("Early stopping.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    hist_df = pd.DataFrame(history)
    hist_df.to_excel(os.path.join(fold_out_dir, "training_history.xlsx"), index=False)

    return model, {
        "best_epoch": best_epoch,
        "best_val_l2_macro_f1": best_val_macro_f1,
        "history": history,
    }

### Hard sibling negatives + hierarchy-aware sibling margin

The training dataset still forces a small number of hard sibling negatives into each candidate list.
The new loss additionally enforces a larger score margin between the gold L2 and same-parent sibling negatives than between the gold L2 and cross-parent negatives.


In [ ]:
# =========================
# 9) Main 5-fold experiment
# =========================
def run_single_ablation(ablation_key: str, mode: str):
    seed_everything(cfg.SEED)

    data, text_col, l1_col, l2_col = load_dataset(cfg)
    all_l2_labels = sorted(data[l2_col].unique().tolist())

    if mode == "full_label":
        cfg.CANDIDATE_K = len(all_l2_labels)
        print(f"Full-label condition: ranking all {cfg.CANDIDATE_K} L2 labels.")

    iso_df, retrieval_card_map, reranker_card_map, label_to_parent, label_name_map = load_iso_cards(cfg, all_l2_labels)

    # The full-label ablation bypasses retrieval completely.
    # It does not load the dense retriever or encode label cards/requirements.
    use_retrieval = mode != "full_label"
    label_ids = all_l2_labels

    if use_retrieval:
        if SentenceTransformer is None:
            raise RuntimeError("sentence-transformers is not installed. Run: pip install sentence-transformers")

        retriever_device = "cuda" if torch.cuda.is_available() else "cpu"
        print("Retriever device:", retriever_device)

        if is_qwen3_embedding_model(cfg.RETRIEVER_MODEL_NAME):
            # Qwen3 is larger than MPNet. fp16 reduces memory on GPU.
            try:
                retriever = SentenceTransformer(
                    cfg.RETRIEVER_MODEL_NAME,
                    device=retriever_device,
                    trust_remote_code=True,
                    model_kwargs={"torch_dtype": torch.float16},
                )
            except TypeError:
                try:
                    retriever = SentenceTransformer(
                        cfg.RETRIEVER_MODEL_NAME,
                        device=retriever_device,
                        model_kwargs={"torch_dtype": torch.float16},
                    )
                except TypeError:
                    retriever = SentenceTransformer(cfg.RETRIEVER_MODEL_NAME, device=retriever_device)
        else:
            retriever = SentenceTransformer(cfg.RETRIEVER_MODEL_NAME, device=retriever_device)

        # Encode retrieval label cards once.
        label_cards = [retrieval_card_map[l2] for l2 in label_ids]
        print("Encoding Qwen3 retrieval label cards (FULL contrastive cards)...")
        print("Retrieval card preview:", str(label_cards[0])[:500])
        print("Reranker short card preview:", str(reranker_card_map[label_ids[0]])[:500])
        label_emb = encode_texts(
            retriever,
            label_cards,
            batch_size=cfg.RETRIEVAL_BATCH_SIZE,
            is_query=False,
            text_type="label_card",
        )
    else:
        retriever = None
        label_emb = None
        print(f"FULL-LABEL BYPASS ACTIVE: retrieval is skipped and all {len(label_ids)} L2 labels are sent to the reranker.")
        print("Reranker short card preview:", str(reranker_card_map[label_ids[0]])[:500])

    # Reuse the exact saved folds; do not call a CV splitter here.
    saved_folds = [
        np.flatnonzero(data["test_fold"].to_numpy() == fold_id).astype(int)
        for fold_id in range(cfg.N_FOLDS)
    ]
    if any(len(indices) == 0 for indices in saved_folds):
        raise ValueError("At least one saved test fold is empty.")
    all_test_indices = np.concatenate(saved_folds)
    if len(all_test_indices) != len(data) or len(np.unique(all_test_indices)) != len(data):
        raise ValueError("Saved test folds must cover every requirement exactly once.")
    print("Validated: saved folds cover every requirement exactly once as test data.")

    all_test_predictions = []
    all_fold_summary = []
    all_retrieval_rows = []

    for fold_id in range(cfg.N_FOLDS):
        print("\n" + "=" * 90)
        print(f"FOLD {fold_id}")
        print("=" * 90)

        # Rotating design: one fold test, next fold validation, remaining folds train.
        test_idx = saved_folds[fold_id]
        val_idx = saved_folds[(fold_id + 1) % cfg.N_FOLDS]
        train_idx = np.setdiff1d(np.arange(len(data)), np.concatenate([test_idx, val_idx]))

        train_df = data.iloc[train_idx].reset_index(drop=True).copy()
        val_df = data.iloc[val_idx].reset_index(drop=True).copy()
        test_df = data.iloc[test_idx].reset_index(drop=True).copy()

        print("Train/Val/Test:", train_df.shape, val_df.shape, test_df.shape)
        print("Train L2 classes:", train_df[l2_col].nunique(), "Val:", val_df[l2_col].nunique(), "Test:", test_df[l2_col].nunique())

        fold_out_dir = os.path.join(cfg.OUT_DIR, f"fold_{fold_id}_listwise")
        if os.path.exists(fold_out_dir):
            shutil.rmtree(fold_out_dir)
        os.makedirs(fold_out_dir, exist_ok=True)

        # Candidate generation.
        if mode == "full_label":
            # True no-retrieval ablation: every row receives the complete L2 label set.
            # No query, label-card, or kNN embeddings are computed.
            train_candidate_map = {i: list(label_ids) for i in range(len(train_df))}
            val_candidate_map = {i: list(label_ids) for i in range(len(val_df))}
            test_candidate_map = {i: list(label_ids) for i in range(len(test_df))}

            assert all(len(v) == len(label_ids) for v in train_candidate_map.values())
            assert all(set(v) == set(label_ids) for v in val_candidate_map.values())
            assert all(set(v) == set(label_ids) for v in test_candidate_map.values())

            print(
                f"Full-label fold {fold_id}: retrieval bypassed; "
                f"all {len(label_ids)} labels are ranked for every train/validation/test example."
            )

            # Retrieval recall is not meaningful for this condition. Record only coverage.
            retrieval_row = {
                "fold": fold_id,
                "ablation": ablation_key,
                "mode": mode,
                "knn_top_n": None,
                "candidate_k": len(label_ids),
                "retrieval_bypassed": True,
                "val_full_label_coverage": 1.0,
                "test_full_label_coverage": 1.0,
                "val_gold_missing": 0.0,
                "test_gold_missing": 0.0,
            }
            all_retrieval_rows.append(retrieval_row)
        else:
            # Retrieval-based ablations: encode requirements and build Top-K candidates.
            # For Qwen3 we use asymmetric retrieval:
            # - train_doc_emb: corpus/document embedding used as kNN database
            # - train_query_emb / val_emb / test_emb: query embeddings
            print("Encoding train corpus texts for kNN database...")
            train_doc_emb = encode_texts(
                retriever,
                train_df[text_col].tolist(),
                batch_size=cfg.RETRIEVAL_BATCH_SIZE,
                is_query=False,
                text_type="train_requirement_document",
            )

            print("Encoding train query texts for train candidate generation...")
            train_query_emb = encode_texts(
                retriever,
                train_df[text_col].tolist(),
                batch_size=cfg.RETRIEVAL_BATCH_SIZE,
                is_query=True,
                text_type="requirement_query",
            )

            print("Encoding val query texts...")
            val_emb = encode_texts(
                retriever,
                val_df[text_col].tolist(),
                batch_size=cfg.RETRIEVAL_BATCH_SIZE,
                is_query=True,
                text_type="requirement_query",
            )

            print("Encoding test query texts...")
            test_emb = encode_texts(
                retriever,
                test_df[text_col].tolist(),
                batch_size=cfg.RETRIEVAL_BATCH_SIZE,
                is_query=True,
                text_type="requirement_query",
            )

            train_l2_list = train_df[l2_col].tolist()
            train_self_positions = list(range(len(train_df)))

            train_candidate_map, _ = build_candidates_for_split(
                query_emb=train_query_emb,
                train_emb=train_doc_emb,
                label_emb=label_emb,
                label_ids=label_ids,
                train_l2=train_l2_list,
                mode=mode,
                candidate_k=cfg.CANDIDATE_K,
                knn_top_n=cfg.KNN_TOP_N,
                iso_weight=cfg.ISO_WEIGHT,
                knn_weight=cfg.KNN_WEIGHT,
                knn_score_mode=cfg.KNN_SCORE_MODE,
                query_train_positions=train_self_positions,
            )
            val_candidate_map, _ = build_candidates_for_split(
                query_emb=val_emb,
                train_emb=train_doc_emb,
                label_emb=label_emb,
                label_ids=label_ids,
                train_l2=train_l2_list,
                mode=mode,
                candidate_k=cfg.CANDIDATE_K,
                knn_top_n=cfg.KNN_TOP_N,
                iso_weight=cfg.ISO_WEIGHT,
                knn_weight=cfg.KNN_WEIGHT,
                knn_score_mode=cfg.KNN_SCORE_MODE,
                query_train_positions=None,
            )
            test_candidate_map, _ = build_candidates_for_split(
                query_emb=test_emb,
                train_emb=train_doc_emb,
                label_emb=label_emb,
                label_ids=label_ids,
                train_l2=train_l2_list,
                mode=mode,
                candidate_k=cfg.CANDIDATE_K,
                knn_top_n=cfg.KNN_TOP_N,
                iso_weight=cfg.ISO_WEIGHT,
                knn_weight=cfg.KNN_WEIGHT,
                knn_score_mode=cfg.KNN_SCORE_MODE,
                query_train_positions=None,
            )

            # Candidate recall is evaluated only for retrieval-based conditions.
            val_retrieval = retrieval_recall_metrics(
                val_df, val_candidate_map, l2_col, cfg.RETRIEVAL_RECALL_KS
            )
            test_retrieval = retrieval_recall_metrics(
                test_df, test_candidate_map, l2_col, cfg.RETRIEVAL_RECALL_KS
            )
            retrieval_row = {
                "fold": fold_id,
                "ablation": ablation_key,
                "mode": mode,
                "knn_top_n": cfg.KNN_TOP_N,
                "candidate_k": cfg.CANDIDATE_K,
                "retrieval_bypassed": False,
            }
            retrieval_row.update({f"val_{k}": v for k, v in val_retrieval.items()})
            retrieval_row.update({f"test_{k}": v for k, v in test_retrieval.items()})
            all_retrieval_rows.append(retrieval_row)
            print("Val retrieval:", val_retrieval)
            print("Test retrieval:", test_retrieval)

        # Tokenizer and model with dropout.
        tokenizer = AutoTokenizer.from_pretrained(cfg.RERANKER_MODEL_NAME, use_fast=True)
        model = create_reranker_model(cfg)

        train_dataset = ListwiseRerankDataset(
            df=train_df,
            candidate_map=train_candidate_map,
            label_card_map=reranker_card_map,
            tokenizer=tokenizer,
            text_col=text_col,
            l2_col=l2_col,
            candidate_k=cfg.CANDIDATE_K,
            max_len=cfg.MAX_LEN,
            force_gold=True,
            all_l2_labels=all_l2_labels,
            label_to_parent=label_to_parent,
            use_hard_sibling_negatives=cfg.USE_HARD_SIBLING_NEGATIVES,
            n_hard_sibling_negatives=cfg.N_HARD_SIBLING_NEGATIVES,
            hard_sibling_strategy=cfg.HARD_SIBLING_STRATEGY,
            random_seed=cfg.SEED + cfg.HARD_SIBLING_SEED_OFFSET + fold_id * 1000,
        )
        val_dataset = ListwiseRerankDataset(
            df=val_df,
            candidate_map=val_candidate_map,
            label_card_map=reranker_card_map,
            tokenizer=tokenizer,
            text_col=text_col,
            l2_col=l2_col,
            candidate_k=cfg.CANDIDATE_K,
            max_len=cfg.MAX_LEN,
            force_gold=False,
            all_l2_labels=all_l2_labels,
            label_to_parent=label_to_parent,
            use_hard_sibling_negatives=False,
        )
        test_dataset = ListwiseRerankDataset(
            df=test_df,
            candidate_map=test_candidate_map,
            label_card_map=reranker_card_map,
            tokenizer=tokenizer,
            text_col=text_col,
            l2_col=l2_col,
            candidate_k=cfg.CANDIDATE_K,
            max_len=cfg.MAX_LEN,
            force_gold=False,
            all_l2_labels=all_l2_labels,
            label_to_parent=label_to_parent,
            use_hard_sibling_negatives=False,
        )

        class_weight_map = None
        if cfg.USE_CLASS_WEIGHTS:
            class_weight_map = make_class_weight_map(train_df, l2_col, all_l2_labels, clip=cfg.CLASS_WEIGHT_CLIP)
            pd.DataFrame([
                {"L2": k, "weight": v, "support": int((train_df[l2_col] == k).sum())}
                for k, v in class_weight_map.items()
            ]).to_excel(os.path.join(fold_out_dir, "class_weights.xlsx"), index=False)

        model, train_info = train_listwise_model(
            model=model,
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            tokenizer=tokenizer,
            fold_out_dir=fold_out_dir,
            all_l2_labels=all_l2_labels,
            label_to_parent=label_to_parent,
            class_weight_map=class_weight_map,
            cfg=cfg,
        )

        # Final validation and test evaluation using the best state.
        val_loader = DataLoader(val_dataset, batch_size=cfg.EVAL_BATCH_SIZE, shuffle=False, collate_fn=listwise_collate_fn)
        test_loader = DataLoader(test_dataset, batch_size=cfg.EVAL_BATCH_SIZE, shuffle=False, collate_fn=listwise_collate_fn)

        val_metrics = evaluate_listwise(model, val_loader, device, all_l2_labels, label_to_parent)
        test_metrics = evaluate_listwise(model, test_loader, device, all_l2_labels, label_to_parent)

        print("Best val L2 macro-F1:", val_metrics["l2_macro_f1"])
        print("Test L2 macro-F1:", test_metrics["l2_macro_f1"])

        # Save fold predictions.
        fold_pred_df = test_df[["global_row_id", text_col, l1_col, l2_col]].copy()
        fold_pred_df = fold_pred_df.rename(columns={text_col: "text", l1_col: "gold_L1", l2_col: "gold_L2"})
        fold_pred_df["fold"] = fold_id
        fold_pred_df["ablation"] = ablation_key
        fold_pred_df["pred_L2"] = test_metrics["y_pred"]
        fold_pred_df["pred_L1"] = [label_to_parent.get(x, "unknown") for x in test_metrics["y_pred"]]
        fold_pred_df["gold_L1_from_map"] = [label_to_parent.get(x, "unknown") for x in test_metrics["y_true"]]
        fold_pred_df["rerank_confidence"] = test_metrics["pred_score"]
        fold_pred_df["candidate_L2"] = test_metrics["pred_candidates"]
        fold_pred_df["correct_L2"] = fold_pred_df["gold_L2"] == fold_pred_df["pred_L2"]
        fold_pred_df["correct_L1"] = fold_pred_df["gold_L1_from_map"] == fold_pred_df["pred_L1"]
        fold_pred_df.to_csv(os.path.join(fold_out_dir, "test_predictions.csv"), index=False)
        all_test_predictions.append(fold_pred_df)

        # Fold summary.
        fold_summary = {
            "fold": fold_id,
            "ablation": ablation_key,
            "mode": mode,
            "knn_top_n": cfg.KNN_TOP_N,
            "candidate_k": cfg.CANDIDATE_K,
            "card_mode": cfg.CARD_TEXT_MODE,
            "best_epoch": train_info["best_epoch"],
            "best_val_l2_macro_f1_during_training": train_info["best_val_l2_macro_f1"],
            "hidden_dropout": cfg.HIDDEN_DROPOUT,
            "attention_dropout": cfg.ATTENTION_DROPOUT,
            "classifier_dropout": cfg.CLASSIFIER_DROPOUT,
            "use_class_weights": cfg.USE_CLASS_WEIGHTS,
            "use_hard_sibling_negatives": cfg.USE_HARD_SIBLING_NEGATIVES,
            "n_hard_sibling_negatives": cfg.N_HARD_SIBLING_NEGATIVES,
            "hard_sibling_strategy": cfg.HARD_SIBLING_STRATEGY,
            "use_l1_aux": cfg.USE_L1_AUX,
            "lambda_l1_aux": cfg.LAMBDA_L1_AUX if cfg.USE_L1_AUX else 0.0,
            "l1_aux_pool": cfg.L1_AUX_POOL,
        }
        fold_summary.update({f"val_{k}": v for k, v in val_metrics.items() if not isinstance(v, list)})
        fold_summary.update({f"test_{k}": v for k, v in test_metrics.items() if not isinstance(v, list)})
        all_fold_summary.append(fold_summary)

        pd.DataFrame([fold_summary]).to_excel(os.path.join(fold_out_dir, "fold_summary.xlsx"), index=False)

        if cfg.DELETE_FOLD_MODEL_AFTER_EVAL and os.path.exists(fold_out_dir):
            # Keep reports, remove large model files only.
            for fn in ["model.safetensors", "pytorch_model.bin", "optimizer.pt", "scheduler.pt"]:
                fp = os.path.join(fold_out_dir, fn)
                if os.path.exists(fp):
                    os.remove(fp)

        # Clean memory.
        del model, tokenizer, train_dataset, val_dataset, test_dataset
        # Free retrieval embeddings only when retrieval was used.
        if mode != "full_label":
            del train_doc_emb, train_query_emb, val_emb, test_emb
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # =========================
    # 10) Save OOF reports
    # =========================
    pred_all = pd.concat(all_test_predictions, ignore_index=True)
    pred_path = os.path.join(cfg.OUT_DIR, "OOF_TEST_listwise_predictions.csv")
    pred_all.to_csv(pred_path, index=False)
    print("Saved:", pred_path)

    summary_df = pd.DataFrame(all_fold_summary)
    summary_path = os.path.join(cfg.OUT_DIR, "OOF_TEST_listwise_summary.xlsx")
    summary_df.to_excel(summary_path, index=False)
    print("Saved:", summary_path)

    retrieval_df = pd.DataFrame(all_retrieval_rows)
    retrieval_path = os.path.join(cfg.OUT_DIR, "retrieval_candidate_recall_by_fold.xlsx")
    retrieval_df.to_excel(retrieval_path, index=False)
    print("Saved:", retrieval_path)

    # Mean/std summary over folds.
    metric_cols = [c for c in summary_df.columns if c.startswith("test_") or c.startswith("val_")]
    numeric_cols = [c for c in metric_cols if pd.api.types.is_numeric_dtype(summary_df[c])]
    mean_std_rows = []
    for c in numeric_cols:
        mean_std_rows.append({"metric": c, "mean": summary_df[c].mean(), "std": summary_df[c].std(ddof=1)})
    mean_std_df = pd.DataFrame(mean_std_rows)
    mean_std_path = os.path.join(cfg.OUT_DIR, "listwise_summary_mean_std_over_folds.xlsx")
    mean_std_df.to_excel(mean_std_path, index=False)
    print("Saved:", mean_std_path)

    # OOF per-class report.
    y_true = pred_all["gold_L2"].tolist()
    y_pred = pred_all["pred_L2"].tolist()
    report = classification_report(y_true, y_pred, labels=all_l2_labels, output_dict=True, zero_division=0)
    per_class = pd.DataFrame(report).T.reset_index().rename(columns={"index": "label"})
    per_class_path = os.path.join(cfg.OUT_DIR, "OOF_TEST_listwise_per_class.xlsx")
    per_class.to_excel(per_class_path, index=False)
    print("Saved:", per_class_path)

    # OOF overall final metrics.
    oof_metrics = compute_final_metrics(y_true, y_pred, all_l2_labels, label_to_parent, prefix="oof_test_")
    oof_metrics_path = os.path.join(cfg.OUT_DIR, "OOF_TEST_overall_metrics.json")
    with open(oof_metrics_path, "w", encoding="utf-8") as f:
        json.dump(oof_metrics, f, indent=2)
    print("Saved:", oof_metrics_path)

    print("\nFinal OOF Test Metrics")
    for k, v in oof_metrics.items():
        print(f"{k}: {v:.4f}")


    return {
        "ablation": ablation_key,
        "out_dir": cfg.OUT_DIR,
        **oof_metrics,
    }


ABLATION_SPECS = {
    "full_label": {
        "display_name": "Ranked full label set (without retrieval)",
        "mode": "full_label",
    },
    "iso_plus_knn": {
        "display_name": "Ranked with retrieval (ISO + kNN)",
        "mode": "iso_plus_knn",
    },
    "iso_only": {
        "display_name": "Ranked with retrieval (ISO only)",
        "mode": "iso_only",
    },
    "knn_only": {
        "display_name": "Ranked with retrieval (kNN only)",
        "mode": "knn_only",
    },
}


def run_ablation_suite():
    base_candidate_k = int(cfg.CANDIDATE_K)
    rows = []

    for key in cfg.RUN_ABLATIONS:
        if key not in ABLATION_SPECS:
            raise ValueError(f"Unknown ablation: {key}")

        spec = ABLATION_SPECS[key]
        mode = spec["mode"]
        cfg.CANDIDATE_K = base_candidate_k
        cfg.OUT_DIR = os.path.join(cfg.BASE_OUT_DIR, key)
        Path(cfg.OUT_DIR).mkdir(parents=True, exist_ok=True)

        metrics_path = os.path.join(cfg.OUT_DIR, "OOF_TEST_overall_metrics.json")

        print("\n" + "#" * 100)
        print(spec["display_name"])
        print("Selected ablation:", key)
        print("Output:", cfg.OUT_DIR)
        print("#" * 100)

        if cfg.SKIP_COMPLETED_ABLATIONS and os.path.exists(metrics_path):
            print("Existing completed metrics found; skipping this run.")
            with open(metrics_path, "r", encoding="utf-8") as f:
                metrics = json.load(f)
            result = {"ablation": key, "out_dir": cfg.OUT_DIR, **metrics}
        else:
            result = run_single_ablation(ablation_key=key, mode=mode)

        result["display_name"] = spec["display_name"]
        result["mode"] = mode
        rows.append(result)

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    comparison_df = pd.DataFrame(rows)
    preferred = [
        "ablation", "display_name", "mode",
        "oof_test_l2_macro_f1", "oof_test_l2_weighted_f1",
        "oof_test_l2_accuracy", "oof_test_l2_balanced_accuracy",
        "oof_test_parent_macro_f1", "oof_test_parent_accuracy",
        "oof_test_hPrecision", "oof_test_hRecall", "oof_test_hF1",
        "out_dir",
    ]
    cols = [c for c in preferred if c in comparison_df.columns]
    cols += [c for c in comparison_df.columns if c not in cols]
    comparison_df = comparison_df[cols]

    xlsx_path = os.path.join(cfg.BASE_OUT_DIR, "ablation_comparison.xlsx")
    csv_path = os.path.join(cfg.BASE_OUT_DIR, "ablation_comparison.csv")
    comparison_df.to_excel(xlsx_path, index=False)
    comparison_df.to_csv(csv_path, index=False)

    print("\nFinal ablation comparison:")
    metric_cols = [c for c in [
        "display_name", "oof_test_l2_macro_f1",
        "oof_test_l2_weighted_f1", "oof_test_l2_accuracy",
        "oof_test_l2_balanced_accuracy", "oof_test_parent_accuracy",
        "oof_test_hF1"
    ] if c in comparison_df.columns]
    print(comparison_df[metric_cols].to_string(index=False))
    print("Saved:", xlsx_path)
    print("Saved:", csv_path)
    return comparison_df


if __name__ == "__main__":
    ablation_results = run_ablation_suite()